In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from sklearn import model_selection
from tqdm import tqdm



In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



Using device: cuda:0


In [3]:
data_path = "../input/cassava-leaf-disease-classification/"
df = pd.read_csv(os.path.join(data_path, "train.csv"))
df_train, df_valid = model_selection.train_test_split(
    df, test_size=0.1, random_state=42, stratify=df.label.values
)



In [4]:
print(df_train.label.value_counts())



label
3    10370
4     2040
2     1882
1     1711
0      845
Name: count, dtype: int64


In [5]:
train_img_dir = os.path.join(data_path, "train_images")
valid_img_dir = os.path.join(data_path, "train_images")  # validation uses same folder

train_image_paths = [
    os.path.join(train_img_dir, img) for img in df_train.image_id.values
]
valid_image_paths = [
    os.path.join(valid_img_dir, img) for img in df_valid.image_id.values
]

train_targets = df_train.label.values
valid_targets = df_valid.label.values




In [6]:
class CassavaDataset(Dataset):
    def __init__(self, image_paths, targets, transform=None):
        self.image_paths = image_paths
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.targets[idx]
        return image, label




In [7]:
input_size = 384
imagenet_stats = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop((input_size, input_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(*imagenet_stats),
    ]
)

valid_transform = transforms.Compose(
    [
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(*imagenet_stats),
    ]
)

cassava_train = CassavaDataset(
    train_image_paths, train_targets, transform=train_transform
)
cassava_valid = CassavaDataset(
    valid_image_paths, valid_targets, transform=valid_transform
)



In [8]:
batch_size = 16
train_loader = DataLoader(
    cassava_train, batch_size=batch_size, shuffle=True, num_workers=2
)
valid_loader = DataLoader(
    cassava_valid, batch_size=batch_size, shuffle=False, num_workers=2
)



In [9]:
# Load ResNeXt50 without attempting to download pretrained weights (offline safe)
model = models.resnext50_32x4d(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 5)
model = model.to(device)



/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)



In [11]:
model.train()
for epoch in range(2):
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/2"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1} loss: {epoch_loss:.4f}")



Epoch 1/2: 100%|██████████| 1053/1053 [02:26<00:00,  7.21it/s]


Epoch 1 loss: 1.2837


Epoch 2/2: 100%|██████████| 1053/1053 [02:26<00:00,  7.18it/s]

Epoch 2 loss: 0.9976


In [12]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in tqdm(valid_loader, desc="Validation"):
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
val_acc = 100 * correct / total
print(f"Validation Accuracy: {val_acc:.2f}%")



Validation: 100%|██████████| 118/118 [00:07<00:00, 15.39it/s]

Validation Accuracy: 62.73%


In [13]:
test_img_dir = os.path.join(data_path, "test_images")
test_files = sorted([f for f in os.listdir(test_img_dir) if f.lower().endswith(".jpg")])
test_paths = [os.path.join(test_img_dir, f) for f in test_files]

test_transform = transforms.Compose(
    [
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(*imagenet_stats),
    ]
)

model.eval()
preds = []
for img_path in tqdm(test_paths, desc="Test Inference"):
    img = Image.open(img_path).convert("RGB")
    img = test_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(img)
        _, pred = torch.max(out, 1)
    preds.append(pred.item())

submission = pd.DataFrame({"image_id": test_files, "label": preds})
submission.head()



Test Inference: 100%|██████████| 2676/2676 [00:29<00:00, 91.74it/s]


,image_id,label
0,1234294272.jpg,3
1,1234332763.jpg,3
2,1234375577.jpg,4
3,1234555380.jpg,4
4,1234571117.jpg,3


In [14]:
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Submission saved to submission.csv
